In [6]:
# ============================================================
# ANOVA + TUKEY HSD
# raw_runs(1).csv
# KB4 - Urgent Success Rate
# ============================================================

import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd


# ============================================================
# CONFIG
# ============================================================

FILE = "/home/shieldx/Documents/files (2)/outputs/raw_runs.csv"

SCENARIO = "KB4_QuaTaiNang"

METRIC = "urgent_success_pct"

ALPHA = 0.05

# So sánh 4 phương pháp
METHODS = [
    "FIFO",
    "Rule-based",
    "TOPSIS",
    "Hybrid"
]

# TOPSIS và Hybrid sử dụng EWM
WEIGHT_METHOD = "AHP"


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv(FILE)

print("=" * 70)
print("DATA INFORMATION")
print("=" * 70)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


# ============================================================
# 2. FILTER KB4
# ============================================================

kb4 = df[
    df["scenario"] == SCENARIO
].copy()


# ============================================================
# 3. LỌC ĐÚNG 4 NHÓM
# ============================================================

# FIFO và Rule-based không có weight method
fifo = kb4[
    kb4["strategy"] == "FIFO"
][METRIC].dropna()

rule = kb4[
    kb4["strategy"] == "Rule-based"
][METRIC].dropna()

# TOPSIS - EWM
topsis = kb4[
    (kb4["strategy"] == "TOPSIS") &
    (kb4["weight_method"] == WEIGHT_METHOD)
][METRIC].dropna()

# Hybrid - EWM
hybrid = kb4[
    (kb4["strategy"] == "Hybrid") &
    (kb4["weight_method"] == WEIGHT_METHOD)
][METRIC].dropna()


# ============================================================
# 4. KIỂM TRA SỐ LƯỢNG OBSERVATION
# ============================================================

groups = {
    "FIFO": fifo,
    "Rule-based": rule,
    "TOPSIS-EWM": topsis,
    "Hybrid-EWM": hybrid
}

print("\n" + "=" * 70)
print("SAMPLE SIZE")
print("=" * 70)

for name, values in groups.items():
    print(
        f"{name:<15} "
        f"n = {len(values)}"
    )


# ============================================================
# 5. MEAN + SD
# ============================================================

print("\n" + "=" * 70)
print("DESCRIPTIVE STATISTICS")
print("=" * 70)

for name, values in groups.items():

    print(
        f"{name:<15} "
        f"Mean = {values.mean():.4f}    "
        f"SD = {values.std(ddof=1):.4f}"
    )


# ============================================================
# 6. ONE-WAY ANOVA
# ============================================================

F, p = f_oneway(
    fifo,
    rule,
    topsis,
    hybrid
)

N = sum(
    len(values)
    for values in groups.values()
)

k = len(groups)

df_between = k - 1
df_within = N - k


# ============================================================
# 7. ANOVA RESULT
# ============================================================

print("\n" + "=" * 70)
print("ONE-WAY ANOVA")
print("=" * 70)

print(
    f"F({df_between}, {df_within}) = {F:.4f}"
)

print(
    f"p = {p:.10e}"
)

if p < ALPHA:
    print(
        "\nConclusion: Có sự khác biệt có ý nghĩa "
        "thống kê giữa các phương pháp."
    )
else:
    print(
        "\nConclusion: Không có sự khác biệt "
        "có ý nghĩa thống kê."
    )


# ============================================================
# 8. TUKEY HSD
# ============================================================

# Tạo dataframe cho Tukey

tukey_df = pd.DataFrame({
    "value": pd.concat(
        [fifo, rule, topsis, hybrid],
        ignore_index=True
    ),

    "method": (
        ["FIFO"] * len(fifo) +
        ["Rule-based"] * len(rule) +
        ["TOPSIS-EWM"] * len(topsis) +
        ["Hybrid-EWM"] * len(hybrid)
    )
})


tukey = pairwise_tukeyhsd(
    endog=tukey_df["value"],
    groups=tukey_df["method"],
    alpha=ALPHA
)


# ============================================================
# 9. HIỂN THỊ TUKEY
# ============================================================

print("\n" + "=" * 70)
print("TUKEY HSD")
print("=" * 70)

print(tukey)


# ============================================================
# 10. XUẤT KẾT QUẢ
# ============================================================

anova_result = pd.DataFrame({
    "Scenario": [SCENARIO],
    "Metric": [METRIC],
    "N": [N],
    "Groups": [k],
    "df_between": [df_between],
    "df_within": [df_within],
    "F": [F],
    "p": [p]
})

anova_result.to_csv(
    "ANOVA_KB4_urgent_success.csv",
    index=False
)


# ============================================================
# 11. TUKEY → CSV
# ============================================================

tukey_table = pd.DataFrame(
    data=tukey._results_table.data[1:],
    columns=tukey._results_table.data[0]
)

tukey_table.to_csv(
    "Tukey_KB4_urgent_success.csv",
    index=False
)


print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print("Đã xuất:")
print("1. ANOVA_KB4_urgent_success.csv")
print("2. Tukey_KB4_urgent_success.csv")

DATA INFORMATION
Shape: (1080, 18)
Columns: ['scenario', 'strategy', 'weight_method', 'seed', 'num_students', 'overload_ratio', 'violation_level', 'rejected_hard', 'waitlist', 'success_rate_pct', 'urgent_success_pct', 'normal_success_pct', 'fairness_index', 'capacity_util_pct', 'filter_ms', 'rank_ms', 'allocate_ms', 'total_ms']

SAMPLE SIZE
FIFO            n = 30
Rule-based      n = 30
TOPSIS-EWM      n = 30
Hybrid-EWM      n = 30

DESCRIPTIVE STATISTICS
FIFO            Mean = 23.2313    SD = 0.9934
Rule-based      Mean = 29.9059    SD = 1.0866
TOPSIS-EWM      Mean = 74.9135    SD = 1.0359
Hybrid-EWM      Mean = 75.0143    SD = 1.0401

ONE-WAY ANOVA
F(3, 116) = 21880.8646
p = 1.7135684141e-159

Conclusion: Có sự khác biệt có ý nghĩa thống kê giữa các phương pháp.

TUKEY HSD
     Multiple Comparison of Means - Tukey HSD, FWER=0.05     
  group1     group2   meandiff p-adj   lower   upper   reject
-------------------------------------------------------------
      FIFO Hybrid-EWM   51.78